# 03 – Train U-Net (Kaggle GPU / Colab GPU)

Trains the residual U-Net for shadow removal on preprocessed SD7K-512.

**Kaggle**:
1. Turn on GPU: in the right sidebar under **Settings** → **Accelerator** → choose **GPU T4 x2** or **P100**.
2. Attach your preprocessed data: click **+ Add Input** → search your dataset (`sd7k-512`) OR select **Notebooks / Your Work** → `02_preprocess_sd7k`.
3. Run All.

**Colab**: Mount Drive or clone repo, set paths, then run.

Folder layout expected inside `sd7k-512`:
```
train/input/    train/target/
val/input/      val/target/
test/input/     test/target/
```

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip -q install segmentation-models-pytorch albumentations pytorch-msssim pyyaml tqdm
import subprocess, sys
if 'google.colab' in sys.modules:
    subprocess.run(['apt-get', '-qq', 'install', 'tesseract-ocr'], check=True)

In [ ]:
# ── Setup, Repository Clone & Path Auto-detection ───────────────────────────
import os, sys, shutil
from pathlib import Path
import torch

KAGGLE = os.path.exists('/kaggle')
COLAB  = 'google.colab' in sys.modules

def find_sd7k_512() -> Path:
    """Auto-detect directory containing train/input and val/input."""
    search_roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('data'), Path('.')]
    for root in search_roots:
        if not root.exists():
            continue
        for candidate in [root] + [p for p in root.rglob('*') if p.is_dir()]:
            try:
                if (candidate / 'train' / 'input').exists() and (candidate / 'val' / 'input').exists():
                    return candidate
            except Exception:
                pass
    return Path('/kaggle/input/sd7k-512') if KAGGLE else Path('data/sd7k_512')

if KAGGLE:
    SD7K_512 = find_sd7k_512()
    CKPT_DIR = Path('/kaggle/working/checkpoints')
    LOG_DIR  = Path('/kaggle/working/logs')
    PROJECT_ROOT = Path('/kaggle/working/DocRescue')

    # Clone repo or copy if uploaded as dataset
    if not PROJECT_ROOT.exists():
        if Path('/kaggle/input/docrescue').exists():
            shutil.copytree('/kaggle/input/docrescue', PROJECT_ROOT)
        else:
            !git clone https://github.com/himanshh-k/DocRescue.git {PROJECT_ROOT}
    sys.path.insert(0, str(PROJECT_ROOT))
elif COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists('/content/drive/MyDrive/DocRescue'):
        PROJECT_ROOT = Path('/content/drive/MyDrive/DocRescue')
    else:
        PROJECT_ROOT = Path('/content/DocRescue')
        if not PROJECT_ROOT.exists():
            !git clone https://github.com/himanshh-k/DocRescue.git {PROJECT_ROOT}
    SD7K_512 = PROJECT_ROOT / 'data' / 'sd7k_512'
    CKPT_DIR = PROJECT_ROOT / 'experiments' / 'checkpoints'
    LOG_DIR  = PROJECT_ROOT / 'experiments' / 'logs'
    sys.path.insert(0, str(PROJECT_ROOT))
else:
    PROJECT_ROOT = Path('.')
    SD7K_512 = Path('data/sd7k_512')
    CKPT_DIR = Path('experiments/checkpoints')
    LOG_DIR  = Path('experiments/logs')
    sys.path.insert(0, '.')

has_gpu = torch.cuda.is_available()
device_name = torch.cuda.get_device_name(0) if has_gpu else 'CPU (⚠️ GPU not enabled in Kaggle Settings!)'
print(f'Device       : {device_name}')
print(f'Project root : {PROJECT_ROOT}')
print(f'SD7K-512     : {SD7K_512}')

if (SD7K_512 / 'train' / 'input').exists():
    n_tr = len(list((SD7K_512 / 'train' / 'input').glob('*')))
    n_val = len(list((SD7K_512 / 'val' / 'input').glob('*')))
    print(f'✅ Dataset verified: {n_tr} train images, {n_val} val images')
else:
    print(f'⚠️ Warning: train/input not found at {SD7K_512}!')
    print('Paths in /kaggle/input:')
    for p in sorted(Path('/kaggle/input').rglob('*'))[:25]: print(f'  {p}')

In [ ]:
# ── Build config dict (override paths for notebook) ────────────────────────
import yaml

config_file = PROJECT_ROOT / 'configs' / 'config.yaml'
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['data']['sd7k_root'] = str(SD7K_512)
cfg['training']['checkpoint_dir'] = str(CKPT_DIR)
cfg['training']['log_dir'] = str(LOG_DIR)

# Tune for your GPU VRAM:
#   T4  (16 GB) → batch 12-16
#   P100 (16 GB) → batch 12-16
#   V100 (32 GB) → batch 24
cfg['training']['batch_size'] = 12
cfg['training']['epochs'] = 50

print(yaml.dump(cfg))

In [ ]:
# ── Run training ────────────────────────────────────────────────────────────
from src.train import train
train(cfg, resume_ckpt=None)  # pass a ckpt path (e.g. str(CKPT_DIR / 'best.pth')) to resume

In [ ]:
# ── TensorBoard ──────────────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

In [ ]:
# ── Quick visual check of best checkpoint ──────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from src.models import build_model
from src.infer import load_checkpoint, gain_map_rescue
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cfg).to(device)
best_path = CKPT_DIR / 'best.pth'
if best_path.exists():
    load_checkpoint(model, str(best_path), device=device)
    print(f'Loaded checkpoint from {best_path}')
else:
    print(f'No checkpoint found at {best_path}')

test_input_dir = SD7K_512 / 'test' / 'input'
test_target_dir = SD7K_512 / 'test' / 'target'
test_files = sorted(os.listdir(test_input_dir))[:3] if test_input_dir.exists() else []

if test_files:
    fig, axes = plt.subplots(len(test_files), 3, figsize=(12, 4 * len(test_files)))
    if len(test_files) == 1:
        axes = np.expand_dims(axes, 0)
    for row, fn in enumerate(test_files):
        inp  = np.array(Image.open(test_input_dir / fn).convert('RGB'))
        tgt  = np.array(Image.open(test_target_dir / fn).convert('RGB'))
        pred = gain_map_rescue(inp, model, device=device)
        axes[row, 0].imshow(inp);  axes[row, 0].set_title('Input');        axes[row, 0].axis('off')
        axes[row, 1].imshow(pred); axes[row, 1].set_title('Predicted');    axes[row, 1].axis('off')
        axes[row, 2].imshow(tgt);  axes[row, 2].set_title('Target (GT)'); axes[row, 2].axis('off')

    plt.suptitle('U-Net output (best checkpoint)', fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print('No test images available to visualize.')